# 1.1.2 — Подстановка реальных данных опыта

**Альтернатива ноутбуку 1.1.** Этот ноутбук собирает артефакт в том же формате, но из
**реальных лабораторных данных** циклических испытаний — без каких-либо синтетических
конструктов. Дальнейшие ноутбуки (1.2 анализ, 1.3 CRR-параметры, 1.4 сплиты, серии 2–3)
работают поверх него без изменений.

## Контракт наблюдаемых данных

Нужны только величины, реально доступные из опыта и паспорта грунта:

| Объект | Форма | Что это |
|---|---|---|
| `soil_df` | таблица (n) | свойства грунта: грансостав + физмех (`e, D_r, I_p, V_s, xi, sigma_eff, permeability, fines_content, clay_fraction, Cu, OCR, K0, Vs1`), `type_ground`/`class_id`, `soil_type` |
| `load_df` | таблица (n) | параметры нагружения: `CSR_base, frequency, amp_scale, N_max, nonstationarity, load_mode, mode_id` |
| `cycles` | (n, T) | число циклов в узлах измерений |
| `csr` | (n, T) | приложенный CSR в узлах |
| `r_measured` | (n, T) | **измеренное** поровое давление PPR(N) |
| `valid_mask` | (n, T) | маска валидной длины измерений (опыт мог закончиться раньше) |
| `liq_label` | (n,) | разжижение: 1/0 |
| `n_liq` | (n,) | число циклов до разжижения (или `N_max`, если не разжижился) |

Параметры кривой CRR (α, β) и факторы вычисляются из свойств грунта физической моделью —
это **входные признаки**, доступные и на реальных данных. Скрытые состояния (z, g, истинная
CRR, непрерывный риск) **не используются**.

**Наблюдаемая супервизия (аналог deep-supervision без синтетики).** Из измеренной кривой
PPR(N) автоматически выводятся вспомогательные цели обучения: мягкий триггер `g_obs` (момент
достижения PPR≈1) и мягкий риск `risk_proxy` (пиковое PPR). Если для грунта есть измеренная
кривая потенциала разжижения CRR(N) (серия из 6 образцов) — её можно подать как `crr_obs` с
по-образцовой маской, и граница CRR модели обучится по реальному измерению там, где оно есть.

## Окружение

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

from liquefaction_ai import get_default_config, save_population_artifact, set_global_seed
from liquefaction_ai.data import build_population_from_experiments, prepare_benchmark_dataset

config = get_default_config()
set_global_seed(config.seed)
# Куда сохранить артефакт. Для замены основного датасета поставьте: OUTPUT_DIR = DATA_DIR
OUTPUT_DIR = REPO_ROOT / "data" / "real_demo"
print("Output:", OUTPUT_DIR)

Output: /sessions/zealous-kind-sagan/mnt/liquefaction-ai/data/real_demo


## Шаг 1. Загрузка реальных массивов

**Замените этот блок на загрузку ваших данных** (например, `np.load(...)`, `pd.read_parquet(...)`).
Ниже — демонстрационная заглушка: берём наблюдаемые массивы из синтетического генератора и
подаём их так, как если бы это были реальные опыты (свойства + измеренная PPR(N) + метки).

In [2]:
# --- ДЕМО-ЗАГЛУШКА (заменить на свои массивы) ---
from dataclasses import replace
from liquefaction_ai import generate_population
demo = generate_population(replace(config, n_scenarios=4000, benchmark_subset=2000))
meta = demo["meta"]
soil_cols = ["class_id", "type_ground", "soil_type", "e", "D_r", "I_p", "V_s", "xi", "sigma_eff",
             "permeability", "fines_content", "clay_fraction", "Cu", "OCR", "K0", "Vs1",
             "static_shear_ratio", "Il", "Ir"]
soil_df = meta[[c for c in soil_cols if c in meta.columns]].copy()
load_df = meta[["CSR_base", "frequency", "amp_scale", "N_max", "nonstationarity", "load_mode", "mode_id"]].copy()
cycles = demo["cycles"]; csr = demo["csr"]
r_measured = demo["r_obs"]            # измеренное поровое давление (с шумом опыта)
valid_mask = demo["valid_mask"]
liq_label = demo["liq_label"]; n_liq = demo["n_liq_true"]
# Опционально: измеренная кривая потенциала разжижения CRR(N) для части грунтов (серия из 6
# образцов). Если у вас её нет — оставьте crr_obs=None.
crr_obs = demo["crr_obs"]; crr_obs_mask = demo["crr_obs_mask"]
# --- КОНЕЦ ДЕМО-ЗАГЛУШКИ ---
print("tests:", len(soil_df), "| PPR shape:", r_measured.shape, "| liq rate:", round(float(liq_label.mean()), 3),
      "| с измеренной CRR:", f"{float(crr_obs_mask.mean())*100:.0f}%")

tests: 4000 | PPR shape: (4000, 72) | liq rate: 0.52 | с измеренной CRR: 26%


/sessions/zealous-kind-sagan/mnt/liquefaction-ai/src/liquefaction_ai/data/synthetic.py:390: RuntimeWarning: overflow encountered in exp
  phi = np.log1p(np.exp(6.0 * (ratio - 0.92))) / 6.0


## Шаг 2. Сборка артефакта из наблюдаемых данных

`build_population_from_experiments` строит признаки (включая CRR α/β и факторы из физической
модели), метаданные и стратифицированное разбиение — в формате, идентичном синтетическому
артефакту, но без латентных полей.

In [3]:
population = build_population_from_experiments(
    soil_df=soil_df, load_df=load_df, cycles=cycles, csr=csr, r_measured=r_measured,
    valid_mask=valid_mask, liq_label=liq_label, n_liq=n_liq, config=config,
    crr_obs=crr_obs, crr_obs_mask=crr_obs_mask)   # crr_obs=None, если нет кривых потенциала
save_population_artifact(OUTPUT_DIR, population, config)

has_latent = any(k in population for k in ["z_true", "g_true", "crr_mix"])
print("Saved to:", OUTPUT_DIR)
print("static_dim:", population["static_features"].shape[1], "| latent (synthetic) fields present:", has_latent)
print("meta columns:", population["meta"].shape[1])

Saved to: /sessions/zealous-kind-sagan/mnt/liquefaction-ai/data/real_demo
static_dim: 34 | latent (synthetic) fields present: False
meta columns: 90


## Шаг 3. Проверка: датасет готов к обучению

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
for split_name in ["train", "val", "test"]:
    s = benchmark[split_name]
    print(f"{split_name:6s}: static={tuple(s['static'].shape)}, seq={tuple(s['seq_in'].shape)}, "
          f"liq_rate={float(s['label'].float().mean()):.3f}")
display(pd.DataFrame({
    "Metric": ["tests", "liquefaction rate", "mean N_liq (cycles)", "median peak PPR", "mean CRR15"],
    "Value": [len(population["meta"]), round(float(population["meta"]["liq_label"].mean()), 4),
              round(float(population["meta"]["N_liq_true"].mean()), 1),
              round(float(population["meta"]["PPR_max_true"].median()), 4),
              round(float(population["meta"]["crr_ref"].mean()), 4)],
}))

train : static=(2800, 34), seq=(2800, 72, 5), liq_rate=0.519
val   : static=(599, 34), seq=(599, 72, 5), liq_rate=0.519
test  : static=(601, 34), seq=(601, 72, 5), liq_rate=0.521


,Metric,Value
0,tests,4000.0000
1,liquefaction rate,0.5195
2,mean N_liq (cycles),521.2000
3,median peak PPR,0.5614
4,mean CRR15,0.2736


## Итог

Артефакт из реальных данных сохранён. Чтобы пустить весь пайплайн (анализ → обучение →
оценка) на реальных данных, либо поставьте `OUTPUT_DIR = DATA_DIR` в этом ноутбуке, либо
скопируйте каталог в `data/demo_run`. Обучение и оценка используют только наблюдаемые
сигналы (измеренный PPR, метку разжижения, N_liq) — никаких синтетических конструктов.